# Symseeker — RFdiffusion on Colab (emergency / interim access)

Use this notebook when you don't have a local GPU and your institution's
cluster (Singularity/SLURM backend) isn't reachable. It runs the exact
same RFdiffusion job you already built and validated in Symseeker — the
contigs, hotspots, and design count are read verbatim from a
`job_manifest.json` that Symseeker generated for you; nothing about the
run is decided here.

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or any
GPU option offered to you), then run the cells in order.

**Steps:** (1) install RFdiffusion once per session → (2) upload the
.zip bundle Symseeker's `colab.prepare_colab_bundle()` produced → (3) run
inference → (4) download `results.zip` and hand it back to Symseeker's
`colab.import_colab_results()`.

## 1. Install RFdiffusion (once per Colab session)

This is the standard published RFdiffusion setup (clone the repo, install
the SE(3)-Transformer package, fetch the model weights). Colab's own
environment changes over time, so if a step here errors, check
RFdiffusion's README (github.com/RosettaCommons/RFdiffusion) for what
changed — the run command in section 3 below will still work unchanged.

In [ ]:
%%bash
if [ ! -d RFdiffusion ]; then
  git clone https://github.com/RosettaCommons/RFdiffusion.git
fi
cd RFdiffusion
mkdir -p models
cd models
# Base model — covers unconditional, scaffolded, and binder design.
# Add the other weight files from the README if your job needs them
# (e.g. Complex_base_ckpt.pt for multi-chain complexes).
[ -f Base_ckpt.pt ] || wget -q http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt
[ -f Complex_base_ckpt.pt ] || wget -q http://files.ipd.uw.edu/pub/RFdiffusion/1befcb9b28e2f778c39ade8bcfb0d1d5/Complex_base_ckpt.pt

In [ ]:
%%bash
cd RFdiffusion
pip install -q -e . > /dev/null
pip install -q dgl -f https://data.dgl.ai/wheels/cu118/repo.html > /dev/null
cd env/SE3Transformer 2>/dev/null && pip install -q -e . > /dev/null || true

## 2. Upload the Symseeker bundle

Upload the `.zip` produced by `colab.prepare_colab_bundle(job)` on your
own machine.

In [ ]:
import os
import shutil
import zipfile
from google.colab import files

uploaded = files.upload()  # pick the *_colab_bundle.zip file
bundle_zip = next(iter(uploaded))

if os.path.exists("bundle"):
    shutil.rmtree("bundle")
with zipfile.ZipFile(bundle_zip) as zf:
    zf.extractall("bundle")

print("Bundle contents:", os.listdir("bundle"))

## 3. Run RFdiffusion

Reads `job_manifest.json`'s `overrides` list — the exact same Hydra
key=value tokens Symseeker's local/Singularity backends would have
built — and runs inference with them, unmodified. Using a Python argv
list (not a shell string) sidesteps any bash-escaping issues with
`contigmap.contigs=[...]`'s brackets and embedded spaces.

In [ ]:
import json
import os
import subprocess

with open("bundle/job_manifest.json") as f:
    manifest = json.load(f)

print("contigs:     ", manifest["contigs"])
print("hotspot_res: ", manifest["hotspot_res"])
print("num_designs: ", manifest["num_designs"])

# inference.output_prefix="output/design" — make sure "output/" exists
# before RFdiffusion tries to write into it.
os.makedirs("bundle/output", exist_ok=True)

os.chdir("bundle")
argv = ["python", "../RFdiffusion/scripts/run_inference.py"] + manifest["overrides"]
result = subprocess.run(argv)
os.chdir("..")

print("RFdiffusion exit code:", result.returncode)

## 4. Download results

Zips `bundle/output/` (the `design_<N>.pdb`/`.trb` files RFdiffusion just
wrote) and downloads it. Hand `results.zip` to
`colab.import_colab_results(job, "results.zip")` back in Symseeker — it
will drop the designs at the same `output_prefix` a local/Singularity run
would have used, so the rest of your pipeline (ProteinMPNN, etc.) doesn't
need to know these came from Colab.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("results", "zip", "bundle/output")
files.download("results.zip")